In [ ]:
import zipfile
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models

#드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#모델정의
class CNNLSTMModel(nn.Module):
    def __init__(self, hidden_dim=128, num_classes=12, lstm_layers=1):
        super(CNNLSTMModel, self).__init__()
        #CNN: ResNet18
        base_model = models.resnet18(weights=None)
        self.cnn_backbone = nn.Sequential(*list(base_model.children())[:-1])  # (B, 512, 1, 1)
        self.cnn_output_dim = 512
        #LSTM
        self.lstm = nn.LSTM(input_size=self.cnn_output_dim,
                            hidden_size=hidden_dim,
                            num_layers=lstm_layers,
                            batch_first=True)
        #FC
        self.classifier = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(-1, C, H, W)  # (B*T, C, H, W)
        features = self.cnn_backbone(x)  # (B*T, 512, 1, 1)
        features = features.view(B, T, -1)  # (B, T, 512)

        lstm_out, _ = self.lstm(features)
        last_hidden = lstm_out[:, -1, :]  # (B, hidden_dim)

        out = self.classifier(last_hidden)  # (B, 12)
        return out

In [ ]:
#zip 파일 압축 해제
zip_path = '/content/drive/MyDrive/Hundred_real.zip'
extract_path = '/content/Hundred_real'

# 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 압축 해제된 경로 확인
os.listdir(extract_path)

In [ ]:
from torchvision import transforms
from PIL import Image
import numpy as np
import torch
import os

#원하는 데이터에 따라 적절히 수정하여 사용
sequence_folder = "/content/real_skeleton_images/skeleton_images/Hundred/Hundred_wrong"
sequence_length = 30

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])
image_files = sorted([
    f for f in os.listdir(sequence_folder)
    if f.startswith("frame_") and f.endswith(".png")
])[:sequence_length]

images = []
for img_name in image_files:
    img_path = os.path.join(sequence_folder, img_name)
    img = Image.open(img_path).convert("RGB")
    images.append(transform(img).numpy())

sequence_tensor = torch.tensor(np.stack(images)).unsqueeze(0)

In [ ]:
model = CNNLSTMModel()
model.load_state_dict(torch.load("cnn_lstm_epoch10.pth", map_location=torch.device('cpu'))) #다운로드 받아 코랩 로컬에 업로드한(테스트해보고자 하는 동작 모델) 파일명으로 적절히 수정하여 사용
model.eval()

In [ ]:
ALL_JOINTS = [
    "RShoulder", "RElbow", "RWrist",
    "LShoulder", "LElbow", "LWrist",
    "RHip", "RKnee", "RAnkle",
    "LHip", "LKnee", "LAnkle"
]
with torch.no_grad():
    output = model(sequence_tensor)
    prediction = (torch.sigmoid(output) > 0.5).int()

print("예측된 잘못된 관절:", [ALL_JOINTS[i] for i, val in enumerate(prediction[0]) if val == 1])